# Fit LG — Twitch community graphs (`twitch`)

Batch fit and model comparison for **all networks** in this collection.

Six MUSAE Twitch community graphs (`data/twitch/graphs_processed/*_graph.edges`), sorted by |V|.

**Per network**, the fit loop logs four steps and saves a 4-panel report (`runs/twitch/{graph}/fit_report.png`):

1. **Load** — graph size and structural attributes  
2. **AIC** — pick `d̂` over candidates `[0,1,2,3]`  
3. **σ̂** — Layer-2 offset logit at `d̂`  
4. **Compare** — LG vs ER / WS / BA (`GraphModelComparator`, spectral GIC)

Reports are also displayed inline (only 6 networks). Aggregate outputs: `runs/twitch/`.

## 1. Discover networks

In [ ]:
import os
import sys
import warnings
from pathlib import Path

for v in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(v, "1")
warnings.filterwarnings("ignore", category=DeprecationWarning)

_REFACTOR = Path.cwd()
if str(_REFACTOR) not in sys.path:
    sys.path.insert(0, str(_REFACTOR))

from platform_fit_utils import PlatformConfig, discover_graph_files, print_discovery

cfg = PlatformConfig(
    platform="twitch",
    glob_pattern="twitch/graphs_processed/*_graph.edges",
    min_nodes=0,
    display_plots=True,  # only 6 networks — show each fit report
)

graph_files = discover_graph_files(cfg)
print_discovery(cfg, graph_files)

## 2. Fit all networks

In [ ]:
from platform_fit_utils import fit_all_networks

comparators, summary_all, fit_meta, failures = fit_all_networks(graph_files, cfg)
RUN_DIR = cfg.run_dir

## 3. Aggregate comparison

In [ ]:
from platform_fit_utils import summarize_aggregates

gic_pivot, rank_pivot, mean_rank = summarize_aggregates(summary_all, RUN_DIR)

## 4. Summary plots

In [ ]:
from platform_fit_utils import plot_aggregate_summary

plot_aggregate_summary(fit_meta, mean_rank, RUN_DIR, cfg.platform, display=True)